# Complete Inference & Evaluation Pipeline
## Preprocess Data -> Load Models -> Evaluate

This notebook downloads datasets, preprocesses them, and evaluates both trained models.

## 1. Setup & Dependencies

In [35]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
import numpy as np
from transformers import AutoTokenizer
import os
import importlib
from huggingface_hub import hf_hub_download

# Hardware optimization
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Device: cuda
CUDA Available: True


## 2. Import Custom Modules

In [36]:
import dataset, model_utils, eval as ev, validate as val, preprocess

importlib.reload(dataset)
importlib.reload(model_utils)
importlib.reload(ev)
importlib.reload(val)
importlib.reload(preprocess)

from dataset import TicketDataset
from model_utils import MultiTaskXLMR

print("Custom modules imported successfully!")

Custom modules imported successfully!


## 3. Define Model Repos from Hugging Face

In [37]:
# Hugging Face repository config
HUGGINGFACE_USERNAME = "pshashid"  # ← CHANGE THIS to your HF username
REPO_NAME_ENGLISH = "xlmr-toxicity-english"
REPO_NAME_MULTILINGUAL = "xlmr-toxicity-multilingual"

ENGLISH_REPO_ID = f"{HUGGINGFACE_USERNAME}/{REPO_NAME_ENGLISH}"
MULTILINGUAL_REPO_ID = f"{HUGGINGFACE_USERNAME}/{REPO_NAME_MULTILINGUAL}"

# Must match the filenames used in upload_model_to_hugging_face.ipynb
ENGLISH_MODEL_FILENAME = "xlmr_toxicity_english.bin"
MULTILINGUAL_MODEL_FILENAME = "xlmr_toxicity_multilingual.bin"

print(f"English repo     : {ENGLISH_REPO_ID}")
print(f"Multilingual repo: {MULTILINGUAL_REPO_ID}")

English repo     : pshashid/xlmr-toxicity-english
Multilingual repo: pshashid/xlmr-toxicity-multilingual


## 4. Preprocess Datasets (via preprocess.py — matches training exactly)

In [38]:
print("===== Preprocessing Jigsaw English Dataset =====")

# preprocess.preprocess_jigsaw() replicates exactly what main.ipynb does:
#   - jigsaw_train.csv : 90% of HF labeled train split (used for training)
#   - jigsaw_val.csv   : 10% of HF labeled train split (used for evaluation below)
#   - jigsaw_test.csv  : original HF test split — labels are NaN, not used for eval
train_df, val_df, test_df_jigsaw = preprocess.preprocess_jigsaw()

print(f"\nSplit sizes:")
print(f"  Train : {len(train_df):,}")
print(f"  Val   : {len(val_df):,}  ← used for English evaluation")
print(f"  Test  : {len(test_df_jigsaw):,}  (unlabeled HF test split, not used for eval)")

===== Preprocessing Jigsaw English Dataset =====
Loading English Jigsaw dataset...
Any 1s in each label column:
 toxic            True
severe_toxic     True
obscene          True
threat           True
insult           True
identity_hate    True
dtype: bool

Number of positive examples in each label column:
 toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Saved Jigsaw preprocessed data to data/processed/jigsaw
Train: 143613, Validation: 15958, Test: 306328

Split sizes:
  Train : 143,613
  Val   : 15,958  ← used for English evaluation
  Test  : 306,328  (unlabeled HF test split, not used for eval)


## 5. Preprocess Multilingual Toxic Dataset

In [39]:
print("===== Preprocessing Multilingual Toxic Dataset =====")

# preprocess.preprocess_multilingual() replicates exactly what main.ipynb does:
#   - merged_non_en_train.csv : 80%, stratified on 'toxic', random_state=42
#   - merged_non_en_test.csv  : 20%, stratified on 'toxic', random_state=42
train_df_multi, test_df_multi = preprocess.preprocess_multilingual()

print(f"\nSplit sizes:")
print(f"  Train : {len(train_df_multi):,}")
print(f"  Test  : {len(test_df_multi):,}  ← used for multilingual evaluation")

===== Preprocessing Multilingual Toxic Dataset =====
Loading TextDetox multilingual toxicity dataset...
Merged non-English multilingual dataset saved:
Train: 53099 rows → data/processed/multilingual_toxic/merged_non_en_train.csv
Test: 13275 rows → data/processed/multilingual_toxic/merged_non_en_test.csv

Split sizes:
  Train : 53,099
  Test  : 13,275  ← used for multilingual evaluation


## 6. Initialize Model

In [40]:
model_name = "xlm-roberta-large"
num_intents = 5  # severe_toxic, obscene, threat, insult, identity_hate

# Create model instance
model = MultiTaskXLMR(model_name=model_name, num_intents=num_intents).to(device)
print(f" Model created: {model_name}")
print(f" Model moved to {device}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model created: xlm-roberta-large
 Model moved to cuda


## 7. Evaluate English Model on Jigsaw Test Set

In [41]:
# ── Load English model from Hugging Face ─────────────────────────────────
print("ENGLISH MODEL EVALUATION")

print("\nDownloading English model from Hugging Face...")
english_model_path = hf_hub_download(
    repo_id=ENGLISH_REPO_ID,
    filename=ENGLISH_MODEL_FILENAME
)
model.load_state_dict(torch.load(english_model_path, map_location=device))
model.to(device)
model.eval()
print(" English model loaded")

# ── Load evaluation data ──────────────────────────────────────────────────
print("\nLoading jigsaw_val.csv for English evaluation...")
en_eval_df = pd.read_csv("data/processed/jigsaw/jigsaw_val.csv", low_memory=False)

# Drop rows with NaN or -1 toxic labels (matches main.ipynb filter: toxic != -1)
en_eval_df = en_eval_df[en_eval_df['toxic'].notna() & (en_eval_df['toxic'] != -1)].copy()

# Sample 5000 — matches main.ipynb: .sample(5000, random_state=42)
if len(en_eval_df) > 5000:
    en_eval_df = en_eval_df.sample(5000, random_state=42)

# Cast all label columns to int to prevent NaN leaking into tensors
intent_cols = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
en_eval_df['toxic'] = en_eval_df['toxic'].astype(int)
for col in intent_cols:
    en_eval_df[col] = en_eval_df[col].astype(int)

en_labels = [
    {'toxic': int(t), 'intents': [int(x) for x in row]}
    for t, row in zip(en_eval_df['toxic'].values, en_eval_df[intent_cols].values)
]

en_eval_ds = TicketDataset(en_eval_df['comment_text'].values, en_labels, model_name=model_name)
en_eval_loader = DataLoader(en_eval_ds, batch_size=64, shuffle=False)

print(f"\n English eval set ready:")
print(f"  Samples : {len(en_eval_df):,}")
print(f"  Toxic   : {en_eval_df['toxic'].sum():,}")
print(f"  Clean   : {(en_eval_df['toxic'] == 0).sum():,}")

ENGLISH MODEL EVALUATION

 English model loaded

Loading jigsaw_val.csv for English evaluation...

 English eval set ready:
  Samples : 5,000
  Toxic   : 470
  Clean   : 4,530


In [42]:
print("TOXICITY CLASSIFICATION METRICS")

report, auc, lang_f1s, cm = ev.run_evaluation(
    model,
    en_eval_loader,
    device,
    lang_list=None
)

print(report)
print(f"GLOBAL AUC-ROC: {auc:.4f}")
print("\nCONFUSION MATRIX:")
print(cm)

TOXICITY CLASSIFICATION METRICS


Evaluating: 100%|██████████| 79/79 [00:15<00:00,  5.12it/s]

              precision    recall  f1-score   support

       Clean       0.98      0.98      0.98      4530
       Toxic       0.80      0.83      0.81       470

    accuracy                           0.96      5000
   macro avg       0.89      0.90      0.90      5000
weighted avg       0.97      0.96      0.96      5000

GLOBAL AUC-ROC: 0.9867

CONFUSION MATRIX:
[[4432   98]
 [  80  390]]


In [43]:
print("INTENT CLASSIFICATION METRICS (ROC-AUC per intent)")

# ev.run_intent_evaluation uses masking for -1 labels and is the
# same logic as val.validate_intents but returns a dict of scores
intent_scores = ev.run_intent_evaluation(model, en_eval_loader, device)

INTENT CLASSIFICATION METRICS (ROC-AUC per intent)


Evaluating Intents: 100%|██████████| 79/79 [00:15<00:00,  5.13it/s]


--- PER-INTENT ROC-AUC SCORES ---
SEVERE_TOXIC        : 0.9907
OBSCENE             : 0.9930
THREAT              : 0.8609
INSULT              : 0.9855
IDENTITY_HATE       : 0.9620


## 8. Evaluate Multilingual Model on Non-English Test Set

In [44]:
# ── Load Multilingual model from Hugging Face ────────────────────────────
print("MULTILINGUAL MODEL EVALUATION")

print("\nDownloading Multilingual model from Hugging Face...")
multilingual_model_path = hf_hub_download(
    repo_id=MULTILINGUAL_REPO_ID,
    filename=MULTILINGUAL_MODEL_FILENAME
)
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)
model.eval()
print(" Multilingual model loaded")

# ── Load full multilingual test set ──────────────────────────────────────
# No sampling cap — main.ipynb evaluates the full 20% test split
print("\nLoading full multilingual test set...")
multi_test_df = pd.read_csv(
    "data/processed/multilingual_toxic/merged_non_en_test.csv"
).fillna("")

# Intents are [0]*5 — multilingual set only has toxicity labels
multi_labels = [{'toxic': int(t), 'intents': [0]*5} for t in multi_test_df['toxic'].values]

multi_test_ds = TicketDataset(multi_test_df['text'].values, multi_labels, model_name=model_name)
multi_test_loader = DataLoader(multi_test_ds, batch_size=64, shuffle=False)

print(f"\n Multilingual eval set ready:")
print(f"  Samples   : {len(multi_test_df):,}")
print(f"  Toxic     : {multi_test_df['toxic'].sum():,}")
print(f"  Clean     : {(multi_test_df['toxic'] == 0).sum():,}")
print(f"  Languages : {multi_test_df['language'].nunique()}")
print(f"  {sorted(multi_test_df['language'].unique())}")

MULTILINGUAL MODEL EVALUATION

 Multilingual model loaded

Loading full multilingual test set...

 Multilingual eval set ready:
  Samples   : 13,275
  Toxic     : 6,601
  Clean     : 6,674
  Languages : 14
  ['am', 'ar', 'de', 'es', 'fr', 'he', 'hi', 'hin', 'it', 'ja', 'ru', 'tt', 'uk', 'zh']


In [45]:
print("MULTILINGUAL TOXICITY CLASSIFICATION METRICS")

report, auc, lang_f1s, cm = ev.run_evaluation(
    model,
    multi_test_loader,
    device,
    lang_list=multi_test_df['language'].values
)

print(report)
print(f"GLOBAL AUC-ROC: {auc:.4f}")

print("\nPER-LANGUAGE F1 SCORES (Ascending Order):")
print("-" * 40)
for lang in sorted(lang_f1s, key=lang_f1s.get):
    print(f"{lang.upper():<15}: {lang_f1s[lang]:.4f}")

print("\nCONFUSION MATRIX:")
print(cm)

MULTILINGUAL TOXICITY CLASSIFICATION METRICS


Evaluating: 100%|██████████| 208/208 [00:38<00:00,  5.40it/s]


              precision    recall  f1-score   support

       Clean       0.88      0.87      0.87      6674
       Toxic       0.87      0.87      0.87      6601

    accuracy                           0.87     13275
   macro avg       0.87      0.87      0.87     13275
weighted avg       0.87      0.87      0.87     13275

GLOBAL AUC-ROC: 0.9460

PER-LANGUAGE F1 SCORES (Ascending Order):
----------------------------------------
ZH             : 0.7591
HIN            : 0.7612
HE             : 0.7820
AR             : 0.7969
AM             : 0.8253
IT             : 0.8494
TT             : 0.8724
JA             : 0.8783
DE             : 0.8816
ES             : 0.8905
UK             : 0.9283
HI             : 0.9294
FR             : 0.9803
RU             : 0.9841

CONFUSION MATRIX:
[[5788  886]
 [ 826 5775]]


## 9. Direct Inference Examples

### 9.1 English Model Inference

In [46]:
# Reload English model for inference examples
model.load_state_dict(torch.load(english_model_path, map_location=device))
model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("ENGLISH MODEL - INFERENCE EXAMPLES")

intent_names = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Example 1: Toxic — threat + insult
test_text_1 = "You are a complete idiot and I will make your life hell!"
inputs_1 = tokenizer(test_text_1, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_1, intent_logits_1 = model(inputs_1['input_ids'], inputs_1['attention_mask'])
tox_prob_1 = torch.sigmoid(tox_logits_1.float()).item()
intent_probs_1 = torch.sigmoid(intent_logits_1.float()).cpu().numpy().flatten()
print(f"\n[Toxic] '{test_text_1}'")
print(f"Toxicity Probability: {tox_prob_1:.4f}")
for name, prob in zip(intent_names, intent_probs_1):
    print(f"  {name:<15}: {prob:.4f}")

# Example 2: Clean
test_text_2 = "Hello, how are you doing today?"
inputs_2 = tokenizer(test_text_2, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_2, intent_logits_2 = model(inputs_2['input_ids'], inputs_2['attention_mask'])
tox_prob_2 = torch.sigmoid(tox_logits_2.float()).item()
intent_probs_2 = torch.sigmoid(intent_logits_2.float()).cpu().numpy().flatten()
print(f"\n[Clean] '{test_text_2}'")
print(f"Toxicity Probability: {tox_prob_2:.4f}")
for name, prob in zip(intent_names, intent_probs_2):
    print(f"  {name:<15}: {prob:.4f}")

ENGLISH MODEL - INFERENCE EXAMPLES

[Toxic] 'You are a complete idiot and I will make your life hell!'
Toxicity Probability: 0.9989
  severe_toxic   : 0.2465
  obscene        : 0.8661
  threat         : 0.1294
  insult         : 0.9553
  identity_hate  : 0.3585

[Clean] 'Hello, how are you doing today?'
Toxicity Probability: 0.0003
  severe_toxic   : 0.0009
  obscene        : 0.0011
  threat         : 0.0008
  insult         : 0.0011
  identity_hate  : 0.0009


### 9.2 Multilingual Model Inference

In [47]:
# Load multilingual model for inference
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)
model.eval()

print("MULTILINGUAL MODEL - INFERENCE EXAMPLES")

# Example 1: German toxic sentence
test_text_de = "Du bist ein Idiot und ich werde dein Leben zur Hölle machen!"  # Toxic in German
inputs_de = tokenizer(test_text_de, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_de, _ = model(inputs_de['input_ids'], inputs_de['attention_mask'])

tox_prob_de = torch.sigmoid(tox_logits_de.float()).item()

print(f"\n[German - Toxic] '{test_text_de}'")
print(f"Toxicity Probability: {tox_prob_de:.4f}")

# Example 2: Spanish clean sentence
test_text_es = "Hola, ¿cómo estás hoy?"  # Clean in Spanish
inputs_es = tokenizer(test_text_es, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_es, _ = model(inputs_es['input_ids'], inputs_es['attention_mask'])

tox_prob_es = torch.sigmoid(tox_logits_es.float()).item()

print(f"\n[Spanish - Clean] '{test_text_es}'")
print(f"Toxicity Probability: {tox_prob_es:.4f}")

# Example 3: French toxic sentence
test_text_fr = "Tu es un idiot et je vais te détruire!"  # Toxic in French
inputs_fr = tokenizer(test_text_fr, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_fr, _ = model(inputs_fr['input_ids'], inputs_fr['attention_mask'])

tox_prob_fr = torch.sigmoid(tox_logits_fr.float()).item()

print(f"\n[French - Toxic] '{test_text_fr}'")
print(f"Toxicity Probability: {tox_prob_fr:.4f}")

# Example 4: Italian clean sentence
test_text_it = "Ciao, come stai oggi?"  # Clean in Italian
inputs_it = tokenizer(test_text_it, return_tensors="pt", padding=True, truncation=True).to(device)

with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=torch.bfloat16):
        tox_logits_it, _ = model(inputs_it['input_ids'], inputs_it['attention_mask'])

tox_prob_it = torch.sigmoid(tox_logits_it.float()).item()

print(f"\n[Italian - Clean] '{test_text_it}'")
print(f"Toxicity Probability: {tox_prob_it:.4f}")

MULTILINGUAL MODEL - INFERENCE EXAMPLES

[German - Toxic] 'Du bist ein Idiot und ich werde dein Leben zur Hölle machen!'
Toxicity Probability: 0.9994

[Spanish - Clean] 'Hola, ¿cómo estás hoy?'
Toxicity Probability: 0.0009

[French - Toxic] 'Tu es un idiot et je vais te détruire!'
Toxicity Probability: 0.9989

[Italian - Clean] 'Ciao, come stai oggi?'
Toxicity Probability: 0.0010


## 10. Batch Inference Helper Function

In [48]:
def batch_inference(texts, model, tokenizer, device, model_type='multilingual'):
    """
    Perform batch inference on multiple texts.

    Args:
        texts: List of text strings
        model: Loaded model
        tokenizer: AutoTokenizer
        device: torch device
        model_type: 'english' or 'multilingual'

    Returns:
        DataFrame with toxicity predictions
    """
    model.eval()
    results = []
    intent_names = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                tox_logits, intent_logits = model(inputs['input_ids'], inputs['attention_mask'])

            tox_prob = torch.sigmoid(tox_logits.float()).item()
            tox_pred = 1 if tox_prob > 0.5 else 0

            result = {
                'text': text,
                'toxicity_prob': tox_prob,
                'is_toxic': tox_pred
            }

            if model_type == 'english':
                intent_probs = torch.sigmoid(intent_logits.float()).cpu().numpy().flatten()
                for name, prob in zip(intent_names, intent_probs):
                    result[name] = prob

            results.append(result)

    return pd.DataFrame(results)

print("Batch inference function defined!")

Batch inference function defined!


In [49]:
test_texts = [
    "You are awesome!",
    "I hate you, you're disgusting!",
    "The weather is nice today.",
    "I'm going to destroy you!",
    "Great job on your presentation!"
]

# Reload multilingual model
model.load_state_dict(torch.load(multilingual_model_path, map_location=device))
model.to(device)

results_df = batch_inference(test_texts, model, tokenizer, device, model_type='multilingual')

print("\nBATCH INFERENCE RESULTS (Multilingual Model)")
print(results_df.to_string())


BATCH INFERENCE RESULTS (Multilingual Model)
                              text  toxicity_prob  is_toxic
0                 You are awesome!       0.001598         0
1   I hate you, you're disgusting!       0.993307         1
2       The weather is nice today.       0.000911         0
3        I'm going to destroy you!       0.951863         1
4  Great job on your presentation!       0.000970         0


## 11. Summary & Results

In [50]:
print("PIPELINE SUMMARY")

print("\n COMPLETED STEPS:")
print("  1. Preprocessed Jigsaw dataset (via preprocess.preprocess_jigsaw)")
print(f"     Train : {len(train_df):,} samples (90% of HF labeled train)")
print(f"     Val   : {len(val_df):,} samples (10% of HF labeled train) ← evaluated")

print("\n  2. Preprocessed Multilingual dataset (via preprocess.preprocess_multilingual)")
print(f"     Train : {len(train_df_multi):,} samples")
print(f"     Test  : {len(test_df_multi):,} samples (full — no sampling cap) ← evaluated")

print("\n  3. Models loaded from Hugging Face")
print(f"     English      : {ENGLISH_REPO_ID}/{ENGLISH_MODEL_FILENAME}")
print(f"     Multilingual : {MULTILINGUAL_REPO_ID}/{MULTILINGUAL_MODEL_FILENAME}")

print("\n  4. Evaluation (matches main.ipynb exactly)")
print("     English      : jigsaw_val.csv, 5000 samples, random_state=42")
print("     Multilingual : merged_non_en_test.csv, full set, no sampling cap")

print("\n  5. Inference examples")
print("     English examples + Multilingual (DE, ES, FR, IT) + batch inference")

print("\nPIPELINE COMPLETE")

PIPELINE SUMMARY

 COMPLETED STEPS:
  1. Preprocessed Jigsaw dataset (via preprocess.preprocess_jigsaw)
     Train : 143,613 samples (90% of HF labeled train)
     Val   : 15,958 samples (10% of HF labeled train) ← evaluated

  2. Preprocessed Multilingual dataset (via preprocess.preprocess_multilingual)
     Train : 53,099 samples
     Test  : 13,275 samples (full — no sampling cap) ← evaluated

  3. Models loaded from Hugging Face
     English      : pshashid/xlmr-toxicity-english/xlmr_toxicity_english.bin
     Multilingual : pshashid/xlmr-toxicity-multilingual/xlmr_toxicity_multilingual.bin

  4. Evaluation (matches main.ipynb exactly)
     English      : jigsaw_val.csv, 5000 samples, random_state=42
     Multilingual : merged_non_en_test.csv, full set, no sampling cap

  5. Inference examples
     English examples + Multilingual (DE, ES, FR, IT) + batch inference

PIPELINE COMPLETE
